<a href="https://colab.research.google.com/github/simonemedeiros/OIBSIP/blob/main/Time_Series_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TimesNet: Weather Forecasting and `top_k` Experiment

This notebook reproduces the TimesNet Weather/720 forecasting configuration and evaluates `top_k = 1, 2, 5`.

Each configuration is executed with seeds 2021–2025, resulting in 15 runs.

In [2]:
# 1. Mount Drive + imports

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import sklearn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 2.Experiment configuration

PROJECT_DIR = Path(
    "/content/drive/MyDrive/TimesNet_Weather_Experiment"
)

REPO_DIR = Path("/content/Time-Series-Library")

LOGS_DIR = PROJECT_DIR / "logs"
METRICS_DIR = PROJECT_DIR / "metrics"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

REGISTRY_FILE = PROJECT_DIR / "experiment_registry.csv"
SUMMARY_FILE = ARTIFACTS_DIR / "experimental_summary.csv"

COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"

TOP_K_VALUES = [1, 2, 5]
SEEDS = [2021, 2022, 2023, 2024, 2025]

for directory in [
    PROJECT_DIR,
    LOGS_DIR,
    METRICS_DIR,
    ARTIFACTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

/content/drive/MyDrive/TimesNet_Weather_Experiment


In [5]:
# 4.Load/Create Experiment Registry

if REGISTRY_FILE.exists():

    registry = pd.read_csv(REGISTRY_FILE)
    print("Existing registry loaded.")

else:

    registry = pd.DataFrame(
        [
            {
                "top_k": top_k,
                "seed": seed,
                "status": "pending",
                "MAE": np.nan,
                "MSE": np.nan,
                "RMSE": np.nan,
                "epoch_time_s": np.nan,
                "log_file": "",
                "metrics_file": "",
            }
            for top_k in TOP_K_VALUES
            for seed in SEEDS
        ]
    )

    registry.to_csv(REGISTRY_FILE, index=False)

    print("New registry created.")

registry

New registry created.


,top_k,seed,status,MAE,MSE,RMSE,epoch_time_s,log_file,metrics_file
0,1,2021,pending,NaN,NaN,NaN,NaN,,
1,1,2022,pending,NaN,NaN,NaN,NaN,,
2,1,2023,pending,NaN,NaN,NaN,NaN,,
3,1,2024,pending,NaN,NaN,NaN,NaN,,
4,1,2025,pending,NaN,NaN,NaN,NaN,,
5,2,2021,pending,NaN,NaN,NaN,NaN,,
6,2,2022,pending,NaN,NaN,NaN,NaN,,
7,2,2023,pending,NaN,NaN,NaN,NaN,,
8,2,2024,pending,NaN,NaN,NaN,NaN,,
9,2,2025,pending,NaN,NaN,NaN,NaN,,


In [6]:
# 5. Helper functions

def experiment_id(top_k, seed):
    return f"k{top_k}_seed{seed}"


def metrics_path(top_k, seed):
    return METRICS_DIR / f"{experiment_id(top_k, seed)}.npy"


def log_path(top_k, seed):
    return LOGS_DIR / f"{experiment_id(top_k, seed)}.log"


def save_registry():
    registry.to_csv(REGISTRY_FILE, index=False)


def is_completed(top_k, seed):
    row = registry[
        (registry["top_k"] == top_k) &
        (registry["seed"] == seed)
    ]

    return (
        not row.empty
        and row.iloc[0]["status"] == "completed"
        and metrics_path(top_k, seed).exists()
    )


def update_registry(top_k, seed, **values):

    mask = (
        (registry["top_k"] == top_k) &
        (registry["seed"] == seed)
    )

    for column, value in values.items():
        registry.loc[mask, column] = value

    save_registry()


def extract_epoch_time(log_text):

    values = re.findall(
        r"Epoch:\s*1\s*cost time:\s*([0-9.]+)",
        log_text
    )

    return float(values[-1]) if values else np.nan

In [8]:
# 6. Check experiment status

experiments = [
    (top_k, seed)
    for top_k in TOP_K_VALUES
    for seed in SEEDS
]

for i, (top_k, seed) in enumerate(experiments, 1):

    status = (
        "COMPLETED"
        if is_completed(top_k, seed)
        else "PENDING"
    )

    print(
        f"{i:02d}/15 | "
        f"top_k={top_k} | "
        f"seed={seed} | "
        f"{status}"
    )

01/15 | top_k=1 | seed=2021 | PENDING
02/15 | top_k=1 | seed=2022 | PENDING
03/15 | top_k=1 | seed=2023 | PENDING
04/15 | top_k=1 | seed=2024 | PENDING
05/15 | top_k=1 | seed=2025 | PENDING
06/15 | top_k=2 | seed=2021 | PENDING
07/15 | top_k=2 | seed=2022 | PENDING
08/15 | top_k=2 | seed=2023 | PENDING
09/15 | top_k=2 | seed=2024 | PENDING
10/15 | top_k=2 | seed=2025 | PENDING
11/15 | top_k=5 | seed=2021 | PENDING
12/15 | top_k=5 | seed=2022 | PENDING
13/15 | top_k=5 | seed=2023 | PENDING
14/15 | top_k=5 | seed=2024 | PENDING
15/15 | top_k=5 | seed=2025 | PENDING


In [9]:
# 7. Repository

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/thuml/Time-Series-Library.git",
            str(REPO_DIR),
        ],
        check=True,
    )

os.chdir(REPO_DIR)

subprocess.run(
    ["git", "checkout", COMMIT],
    check=True,
)

current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

assert current_commit == COMMIT

print("Repository ready:", current_commit[:7])

Repository ready: 4e938a1


In [10]:
# 8. Dependencies

!pip install -q einops reformer-pytorch sympy

In [20]:
# 9. Environment

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

print("=== Dependências ===")
!grep -nE 'timesfm|torch|numpy|pandas|scikit-learn|matplotlib|einops|reformer-pytorch|sktime|tqdm' requirements.txt

print("\n=== Script oficial Weather ===")
!cat scripts/long_term_forecast/Weather_script/TimesNet.sh

print("\n=== Imports do TimesNet ===")
!sed -n '1,100p' models/TimesNet.py

Python: 3.13.16
PyTorch: 2.11.0+cpu
CUDA: None
GPU: CPU
NumPy: 2.1.3
Pandas: 2.2.3
scikit-learn: 1.6.1
=== Dependências ===
1:einops==0.8.1
3:reformer-pytorch==1.4.4
5:numpy==2.1.2
7:scikit-learn==1.7.2
8:pandas==2.3.3
9:matplotlib==3.10.8
10:sktime==0.40.1
15:tqdm==4.66.5
22:timesfm==1.3.0

=== Script oficial Weather ===
export CUDA_VISIBLE_DEVICES=5

model_name=TimesNet

python -u run.py \
  --task_name long_term_forecast \
  --is_training 1 \
  --root_path ./dataset/weather/ \
  --data_path weather.csv \
  --model_id weather_96_96 \
  --model $model_name \
  --data custom \
  --features M \
  --seq_len 96 \
  --label_len 48 \
  --pred_len 96 \
  --e_layers 2 \
  --d_layers 1 \
  --factor 3 \
  --enc_in 21 \
  --dec_in 21 \
  --c_out 21 \
  --d_model 32 \
  --d_ff 32 \
  --top_k 5 \
  --des 'Exp' \
  --itr 1

python -u run.py \
  --task_name long_term_forecast \
  --is_training 1 \
  --root_path ./dataset/weather/ \
  --data_path weather.csv \
  --model_id weather_96_192 \
  --model 

In [21]:
%cd /content/Time-Series-Library

!grep -nE 'timesfm|reformer|sktime|einops|numpy|pandas|scikit-learn' requirements.txt
!python -c "import torch, numpy, pandas, sklearn, einops; print('Imports básicos OK')"

/content
1:einops==0.8.1
3:reformer-pytorch==1.4.4
5:numpy==2.1.2
7:scikit-learn==1.7.2
8:pandas==2.3.3
10:sktime==0.40.1
22:timesfm==1.3.0
Imports básicos OK


In [22]:
%cd /content/Time-Series-Library
!python -c "import reformer_pytorch, sktime; print('Dependências restantes OK')"

/content
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import reformer_pytorch, sktime; print('Dependências restantes OK')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'sktime'


In [23]:
%cd /content/Time-Series-Library
!python -c "import sktime; print('sktime OK')"
!grep -R -n "reformer_pytorch" models layers

/content
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import sktime; print('sktime OK')
    ^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'sktime'
layers/SelfAttention_Family.py:6:from reformer_pytorch import LSHSelfAttention


In [24]:
%pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.5 MB/s eta 0:00:00


In [25]:
import reformer_pytorch, sktime
print("Dependências OK")

Dependências OK


In [26]:
%cd /content/Time-Series-Library
!grep -nE 'pred_len|train_epochs|batch_size|top_k|learning_rate|patience' scripts/long_term_forecast/Weather_script/TimesNet.sh

/content
16:  --pred_len 96 \
25:  --top_k 5 \
40:  --pred_len 192 \
49:  --top_k 5 \
52:  --train_epochs 1
66:  --pred_len 336 \
75:  --top_k 5 \
90:  --pred_len 720 \
99:  --top_k 5 \
102:  --train_epochs 1


In [27]:
%pip install patool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 7.1 MB/s eta 0:00:00


In [15]:
# 10. Dataset

!pip -q install gdown
!gdown --id 1nXdMIJ7K201Bx3IBGNiaNFQ6FzeDEzIr -O /content/weather_download
!file /content/weather_download
!ls -lh /content/weather_download

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1nXdMIJ7K201Bx3IBGNiaNFQ6FzeDEzIr
To: /content/weather_download
100% 2.29M/2.29M [00:00<00:00, 32.5MB/s]
/content/weather_download: Zip archive data, at least v2.0 to extract, compression method=store
-rw-r--r-- 1 root root 2.2M Feb 11  2023 /content/weather_download


In [16]:
!unzip -o /content/weather_download -d /content/weather_data
!find /content/weather_data -maxdepth 3 -type f

Archive:  /content/weather_download
   creating: /content/weather_data/weather/
  inflating: /content/weather_data/weather/weather.csv  
/content/weather_data/weather/weather.csv


In [17]:
data_path = "/content/weather_data/weather/weather.csv"
df = pd.read_csv(data_path)

print("Dimensões:", df.shape)
print("\nColunas:")
print(df.columns.tolist())
print("\nPrimeiras linhas:")
display(df.head())
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores ausentes:", df.isna().sum().sum())

Dimensões: (52696, 22)

Colunas:
['date', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'OT']

Primeiras linhas:


,date,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),...,wv (m/s),max. wv (m/s),wd (deg),rain (mm),raining (s),SWDR (W/m�),PAR (�mol/m�/s),max. PAR (�mol/m�/s),Tlog (degC),OT
0,2020-01-01 00:10:00,1008.89,0.71,273.18,-1.33,86.1,6.43,5.54,0.89,3.42,...,1.02,1.60,224.3,0.0,0.0,0.0,0.0,0.0,11.45,428.1
1,2020-01-01 00:20:00,1008.76,0.75,273.22,-1.44,85.2,6.45,5.49,0.95,3.39,...,0.43,0.84,206.8,0.0,0.0,0.0,0.0,0.0,11.51,428.0
2,2020-01-01 00:30:00,1008.66,0.73,273.21,-1.48,85.1,6.44,5.48,0.96,3.39,...,0.61,1.48,197.1,0.0,0.0,0.0,0.0,0.0,11.60,427.6
3,2020-01-01 00:40:00,1008.64,0.37,272.86,-1.64,86.3,6.27,5.41,0.86,3.35,...,1.11,1.48,206.4,0.0,0.0,0.0,0.0,0.0,11.70,430.0
4,2020-01-01 00:50:00,1008.61,0.33,272.82,-1.50,87.4,6.26,5.47,0.79,3.38,...,0.49,1.40,209.6,0.0,0.0,0.0,0.0,0.0,11.81,432.2



Tipos de dados:
date                     object
p (mbar)                float64
T (degC)                float64
Tpot (K)                float64
Tdew (degC)             float64
rh (%)                  float64
VPmax (mbar)            float64
VPact (mbar)            float64
VPdef (mbar)            float64
sh (g/kg)               float64
H2OC (mmol/mol)         float64
rho (g/m**3)            float64
wv (m/s)                float64
max. wv (m/s)           float64
wd (deg)                float64
rain (mm)               float64
raining (s)             float64
SWDR (W/m�)             float64
PAR (�mol/m�/s)         float64
max. PAR (�mol/m�/s)    float64
Tlog (degC)             float64
OT                      float64
dtype: object

Valores ausentes: 0


## Seed Control

`run.py` is modified to accept an experimental seed. The patch is applied only once per runtime.

In [18]:
# 11. Patch run.py

run_file = REPO_DIR / "run.py"
text = run_file.read_text()

if "--seed" not in text:

    seed_argument = """
parser.add_argument('--seed', type=int, default=2021,
                    help='random seed for experimental runs')
"""

    text = text.replace(
        "args = parser.parse_args()",
        seed_argument + "\nargs = parser.parse_args()"
    )

    old_code = """fix_seed = 2021
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)"""

    new_code = """random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

print(f"Experimental seed: {args.seed}")"""

    if old_code not in text:
        raise RuntimeError("Original seed block not found.")

    text = text.replace(old_code, new_code)

    run_file.write_text(text)

assert "--seed" in run_file.read_text()

print("Seed control ready.")

Seed control ready.


In [ ]:
os.chdir(REPO_DIR)

for i, (top_k, seed) in enumerate(experiments, 1):

    if is_completed(top_k, seed):
        print(
            f"{i:02d}/15 | SKIP | "
            f"top_k={top_k} | seed={seed}"
        )
        continue

    print(
        f"\n{'=' * 60}\n"
        f"{i:02d}/15 | top_k={top_k} | seed={seed}\n"
        f"{'=' * 60}"
    )

    model_id = f"weather_96_720_k{top_k}_seed{seed}"

    update_registry(
        top_k,
        seed,
        status="running"
    )

    command = [
        "python", "-u", "run.py",
        "--task_name", "long_term_forecast",
        "--is_training", "1",
        "--root_path", "./dataset/weather/",
        "--data_path", "weather.csv",
        "--model_id", model_id,
        "--model", "TimesNet",
        "--data", "custom",
        "--features", "M",
        "--seq_len", "96",
        "--label_len", "48",
        "--pred_len", "720",
        "--e_layers", "2",
        "--d_layers", "1",
        "--factor", "3",
        "--enc_in", "21",
        "--dec_in", "21",
        "--c_out", "21",
        "--d_model", "32",
        "--d_ff", "32",
        "--top_k", str(top_k),
        "--des", "Exp",
        "--itr", "1",
        "--train_epochs", "1",
        "--batch_size", "32",
        "--num_workers", "0",
        "--seed", str(seed),
    ]

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []

    for line in process.stdout:
        print(line, end="")
        lines.append(line)

    return_code = process.wait()
    log_text = "".join(lines)

    saved_log = log_path(top_k, seed)
    saved_log.write_text(log_text)

    if return_code != 0:

        update_registry(
            top_k,
            seed,
            status="failed",
            log_file=str(saved_log),
        )

        print("Run failed. Log saved.")
        continue

    result_dirs = list(
        (REPO_DIR / "results").glob(
            f"long_term_forecast_{model_id}_TimesNet_*"
        )
    )

    metric_files = [
        path / "metrics.npy"
        for path in result_dirs
        if (path / "metrics.npy").exists()
    ]

    if not metric_files:

        update_registry(
            top_k,
            seed,
            status="failed",
            log_file=str(saved_log),
        )

        print("metrics.npy not found.")
        continue

    saved_metrics = metrics_path(top_k, seed)

    shutil.copy2(
        metric_files[0],
        saved_metrics,
    )

    metrics = np.load(saved_metrics)

    mae = float(metrics[0])
    mse = float(metrics[1])
    rmse = float(np.sqrt(mse))
    epoch_time = extract_epoch_time(log_text)

    update_registry(
        top_k,
        seed,
        status="completed",
        MAE=mae,
        MSE=mse,
        RMSE=rmse,
        epoch_time_s=epoch_time,
        log_file=str(saved_log),
        metrics_file=str(saved_metrics),
    )

    print(
        f"\nSaved | "
        f"MAE={mae:.6f} | "
        f"MSE={mse:.6f} | "
        f"RMSE={rmse:.6f} | "
        f"time={epoch_time:.2f}s"
    )


01/15 | top_k=1 | seed=2021
Using cpu or mps
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           weather_96_720_k1_seed2021Model:              TimesNet            

Data Loader
  Data:               custom              Root Path:          ./dataset/weather/  
  Data Path:          weather.csv         Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            96                  Label Len:          48                  
  Pred Len:           720                 Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              1                   Num Kernels:        6                   
  Enc In:             21                  Dec In:             21                  
  C Out:              21         

In [ ]:
# 13. Results

registry = pd.read_csv(REGISTRY_FILE)

results = (
    registry[
        registry["status"] == "completed"
    ]
    .sort_values(["top_k", "seed"])
    .reset_index(drop=True)
)

print(f"Completed: {len(results)}/15")

display(
    results[
        [
            "top_k",
            "seed",
            "MAE",
            "MSE",
            "RMSE",
            "epoch_time_s",
        ]
    ]
)

In [ ]:
# 14. Summary

summary = (
    results
    .groupby("top_k")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        MSE_mean=("MSE", "mean"),
        MSE_std=("MSE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        time_mean_s=("epoch_time_s", "mean"),
        time_std_s=("epoch_time_s", "std"),
    )
)

display(summary.round(6))

In [ ]:
# 15. Save Summary

summary.to_csv(SUMMARY_FILE)

print("Registry:", REGISTRY_FILE)
print("Summary:", SUMMARY_FILE)
print("Logs:", len(list(LOGS_DIR.glob("*.log"))))
print("Metrics:", len(list(METRICS_DIR.glob("*.npy"))))

## Preliminary Observation

The three `top_k` configurations produced similar predictive results across the evaluated seeds.

`top_k=2` obtained the lowest observed mean predictive errors, but the differences between configurations were small and the standard deviations overlapped.

Larger differences were observed in training epoch time, motivating further analysis of the relationship between frequency selection and computational cost.